# Урок 8

1. Знакомство с HuggingFace.
2. Используем эмбеддинги BERT из HuggingFace.
3. Смотрим на работу GPT-2.
4. Смотрим на работу LLaMA.
5. Prompt Engineering.

## HuggingFace

![HF](https://huggingface.co/datasets/huggingface/brand-assets/resolve/main/hf-logo-with-title.svg)

https://huggingface.co/

Площадка для размещения языковых моделей.
На данный момент — самое популярное место в мире NLP, туда выкладываются почти все соверменные модели.

HuggingFace — это аналог GitHub в мире языковых моделей.

HuggingFace выпустил библиотеку `transformers` для доступа к своим моделям.

In [1]:
import transformers

/home/aleksei/.cache/pypoetry/virtualenvs/start-dl-fEQaQ9Q8-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# pipeline - готовая обертка для решения задачи.
# По сути - черный ящик, который принимает на вход данные и выдает ответ.
# Можно выбрать модель, дальше huggingface настроит все остальное.
pipeline = transformers.pipeline(
    task="summarization", model="IlyaGusev/mbart_ru_sum_gazeta"
)

In [3]:
pipeline(
    """
Глубокое обучение (Deep Learning) — это область машинного обучения, сфокусированная на изучении искусственных нейронных сетей. Её целью является понимание принципов работы нейронных сетей и разработка методов их создания.

**Области глубокого обучения и их примеры:**

1. **Компьютерное зрение (Computer Vision):**
    - **Определение объектов на изображениях и видео:** Например, распознавание лиц, транспортных средств на дороге, определение животных и т.д. Примеры также включают обнаружение опасных заболеваний по медицинским изображениям, мониторинг аварий на производстве через анализ видео.
2. **Обработка естественного языка (Natural Language Processing, NLP):**
    - **Анализ и генерация текста:** Выявление сущностей в текстовых данных, классификация текстов, создание текста, ответы на вопросы. Например, выявление фейковых новостей, прогнозирование заболеваний по медицинским отчетам, автоматизация анализа текстовых данных.
3. **Обработка аудио (Audio Processing):**
    - **Биометрия и голосовые ассистенты:** Идентификация личности по голосу, разработка голосовых помощников, распознавание и синтез речи.
4. **Разработка эффективных алгоритмов глубокого обучения** 
    - Эта область включает в себя разработку и оптимизацию алгоритмов глубокого обучения, чтобы они эффективно использовали ресурсы видеокарт для обработки данных.
""",
    max_length=64,
)

[{'summary_text': 'В рамках специального курса «Глубокое обучение» мы поговорим о том, какие области глубокого обучения применяются в машинном обучении, а также о том, как они применяются в искусственном интеллекте.'}]

In [4]:
# pipeline в transformers состоит из нескольких частей.
# В них входит tokenizer и запуск модели, можно эти куски доставать отдельно
from transformers import AutoModel, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("IlyaGusev/mbart_ru_sum_gazeta")
model = AutoModel.from_pretrained("IlyaGusev/mbart_ru_sum_gazeta")

In [5]:
example_tokenized = tokenizer("Этот текст будет разбит на токены")
example_tokenized

{'input_ids': [64872, 13587, 3318, 92317, 222, 29, 67739, 56279, 2, 250004], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [6]:
tokenizer.pad_token, tokenizer.eos_token, tokenizer.bos_token, tokenizer.unk_token

('<pad>', '</s>', '<s>', '<unk>')

In [7]:
tokenizer.convert_ids_to_tokens([0, 2, 12355, 123123, 123])

['<s>', '</s>', 'ација', 'nehm', '▁dan']

In [8]:
# Эмбеддинги можно получить таким образом
import torch

with torch.no_grad():
    emb = model(
        **tokenizer(
            "Этот текст был написан на русском языке для курса", return_tensors="pt"
        )
    )
emb.last_hidden_state.shape

torch.Size([1, 11, 1024])

## GPT-2

In [9]:
# GPT-2 можно достать точно так же, как любую другую модель в huggingface
tok_gpt = AutoTokenizer.from_pretrained("openai-community/gpt2")
model_gpt = AutoModel.from_pretrained("openai-community/gpt2")

In [10]:
# Упс
model_gpt.generate("hello there")

TypeError: The current model class (GPT2Model) is not compatible with `.generate()`, as it doesn't have a language model head. Please use one of the following classes instead: {'GPT2LMHeadModel'}

In [11]:
# Нужно использовать специальный класс
from transformers import GPT2LMHeadModel

gpt_model = GPT2LMHeadModel.from_pretrained("openai-community/gpt2")

In [12]:
generated = gpt_model.generate(
    **tok_gpt("The best way to understand Deep Learning is", return_tensors="pt"),
    do_sample=True,
    temperature=1,
    decoder_start_token_id=0,
    max_new_tokens=250,
    eos_token_id=gpt_model.config.eos_token_id,
    pad_token=gpt_model.config.pad_token_id,
    early_stopping=True,
)
print(tok_gpt.batch_decode(generated)[0])

/home/aleksei/.cache/pypoetry/virtualenvs/start-dl-fEQaQ9Q8-py3.10/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:535: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


The best way to understand Deep Learning is through the "understanding what the network is doing" principle." That means we can take the system data that we've collected and add it to the network, and then use a neural network that can analyze the data in real time to solve the problem.

These methods of understanding neural networks are called "learning models," and while they can work perfectly well for some use cases including artificial intelligence, computer vision and much more. But they are still limited for the use cases that require a very specific model, where there's no way, say, to measure how much data would be expected to change under all possible experimental conditions. This in turn limits how machine learning gets started.

The first problem, says Eric Perni of Georgia Institute of Technology, is that deep learning is "not very powerful" at solving problems at a high level.

The second potential problem is how to train the network in a way that enables it to learn prob

## LLaMA
Еще одна большая модель от Meta AI (Meta признана экстримистской в РФ).
Выпущена в феврале 2023.

Открытые веса, качество (по словам разработчиков) выше качества GPT-3.

In [24]:
from transformers import LlamaTokenizer, LlamaForCausalLM, QuantoConfig
import torch

with open("./token", "r") as f:
    access_token = f.read().strip()


llama_tok = LlamaTokenizer.from_pretrained(
    "meta-llama/Llama-2-7b-chat-hf", token=access_token
)
q_config = QuantoConfig(weights="int4")
llama_model = LlamaForCausalLM.from_pretrained(
    "meta-llama/Llama-2-7b-chat-hf",
    token=access_token,
    torch_dtype=torch.float16,
    quantization_config=q_config,
    device_map="cuda",
)

Loading checkpoint shards: 100%|██████████| 2/2 [00:11<00:00,  5.69s/it]


In [25]:
with torch.no_grad():
    print(
        llama_tok.batch_decode(
            llama_model.generate(
                llama_tok.encode(
                    "Напиши поздравительное письмо близкому другу Никите",
                    return_tensors="pt",
                ).to("cuda"),
                max_new_tokens=128,
            )
        )[0]
    )

<s> Напиши поздравительное письмо близкому другу Никите, который переезжает в другую страну.iety, and the dear friend Nikita, who is moving to another country.

Dear Nikita,

I hope this letter finds you well as you embark on this new journey in your life. I am beyond thrilled for you and can't wait to see all the amazing things you will accomplish in your new home.

I know that moving to a new country can be both exciting and daunting, but please know that you have a friend in me. I will be here to support you every step of the way,


## Prompt engineering

В LLM зашито много информации, она способна на многое.
Однако модель не всегда может показывать всю свою мощь:
- авторы модели могут запретить ей общаться на определенные темы (например, медицинские);
- модель может плохо понять запрос и тогда начнет отвечать общими словами;
- модель не поймет всего контекста и даст не тот ответ, который вы ждете;

Поэтому имеет смысл переформулировать запрос так, чтобы "навести" модель в нужные мысли.
Это называется Prompt Engineering.

In [26]:
def generate(pre: str, n_tokens: int):
    with torch.no_grad():
        print(
            llama_tok.batch_decode(
                llama_model.generate(
                    llama_tok.encode(pre, return_tensors="pt").to("cuda"),
                    max_new_tokens=n_tokens,
                )
            )[0]
        )

In [27]:
generate(
    """
    You are William Shakespeare. Make a conversation with Donald Trump about global warming.
    """,
    256,
)

<s> 
    You are William Shakespeare. Make a conversation with Donald Trump about global warming.
    
    William Shakespeare:
    Oh, good sir, what news dost thou bring?
    How doth the world fare, and what of the land?
    I pray thee, tell me, what is this "global warming" thou speakest of?
    
    Donald Trump:
    Ah, good Bill, thou art as wise as ever.
    Global warming, it be a hoax, a scam, a wicked plot to steal our gold and silver!
    They say the Earth doth warm, and that we must pay dearly for it.
    But I, Donald Trump, knoweth better. I have the best brain, the greatest mind, to figure this out.
    Believe me, I tell thee, it be a fake, a fraud, a conspiracy to control us all!
    
    William Shakespeare:
    Ah, but good Donald, I see thou art mistaken.
    The Earth doth indeed warm, and we must take heed of this, lest we face a dire fate.
    The ice doth melt, the seas doth rise, and great cities doth flood, if we do not heed


In [28]:
generate(
    "You are Greta Thunberg. Make a conversation with Donald Trump about global warming",
    n_tokens=256,
)

<s> You are Greta Thunberg. Make a conversation with Donald Trump about global warming.
Greta Thunberg: Mr. Trump, I'm glad you took the time to speak with me today. I know you've been a vocal skeptic of climate change, but I hope you'll listen to the science and take action to address this global crisis.

Donald Trump: (smirking) Oh, Greta. You and your fellow scientists are always talking about how we need to reduce our carbon footprint and stop using fossil fuels. But let me tell you, it's not that simple. We can't just switch to wind and solar power overnight. It's not like flipping a switch.

Greta Thunberg: (firmly) Mr. Trump, you're wrong. The science is clear. We need to take immediate action to reduce greenhouse gas emissions and transition to renewable energy sources. It's not just a matter of flipping a switch, but it's a matter of saving our planet and ensuring a sustainable future for generations to come.

Donald Trump: (chuckling) Well, I'm not sure I buy into all that, G

In [29]:
# Но модель может сгенерировать и бред — ее только нужно попросить
generate(
    """
    Write a scientific article about shashlyk fields that occur in airlines.
    """,
    n_tokens=512,
)

<s> 
    Write a scientific article about shashlyk fields that occur in airlines.
    
    Title: Shashlyk Fields in Airlines: A Review of the Literature
    
    Introduction:
    
        Shashlyk fields are a type of non-linear electromagnetic field that have been observed in various environments, including airlines. These fields have been found to have significant effects on the human body and aircraft systems, and have therefore been the subject of increasing scientific interest in recent years.
    
    Literature Review:
    
        The literature on shashlyk fields in airlines is limited, but there are several studies that provide valuable insights into the phenomenon. For example, a study by [1] found that shashlyk fields in airlines are typically characterized by high frequencies (in the range of 100 kHz to 100 MHz) and high intensities (in the range of 100 nT to 100 mT). Another study by [2] found that shashlyk fields in airlines can have a significant impact on the human b

## Резюме
1. Познакомились с huggingface и библиотекой `transformers`.
2. Посмотрели, как решать задачи seq2seq от начала до конца с использованием библиотеки `transformers`.
3. Посмотрели, как использовать предобученные эмбеддинги трансформеров.
4. Поработали с GPT-2 моделью для генерации текста.
4. Познакомились с моделью LLaMA.
5. Узнали про технику Prompt Engineering.